In [ ]:
# libraries
import os
import sys
import json
import hashlib
import platform
from pathlib import Path
import importlib.metadata
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw, ImageOps
from IPython.display import display

In [ ]:
# experiment settings; same function for notebook preparation and CLI submission
def experiment_settings(profile, requested_device, resume=False):
    SEED = 63
    PROFILES = {
        'smoke': {'grid': 24, 'sprite': 10, 'shift': 2, 'state_dim': 8, 'parameter_target': 1200,
                  'batch_size': 2, 'pool_size': 8, 'steps1': 4, 'steps2': 4, 'rollout_min': 4, 'rollout_max': 6,
                  'eval_steps': 6, 'val_every': 2, 'replicates': 1, 'eval_cases': 3, 'latent_samples': 4,
                  'channel_trials': 1, 'val_trials': 1, 'calibration_ps': [0.0, 0.5, 1.0]},
        'pilot': {'grid': 32, 'sprite': 14, 'shift': 3, 'state_dim': 16, 'parameter_target': 6000,
                  'batch_size': 4, 'pool_size': 32, 'steps1': 600, 'steps2': 900, 'rollout_min': 24, 'rollout_max': 40,
                  'eval_steps': 48, 'val_every': 100, 'replicates': 1, 'eval_cases': 12, 'latent_samples': 16,
                  'channel_trials': 2, 'val_trials': 2, 'calibration_ps': [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]},
        'full': {'grid': 40, 'sprite': 18, 'shift': 4, 'state_dim': 16, 'parameter_target': 12000,
                 'batch_size': 8, 'pool_size': 128, 'steps1': 4000, 'steps2': 6000, 'rollout_min': 48, 'rollout_max': 72,
                 'eval_steps': 96, 'val_every': 200, 'replicates': 3, 'eval_cases': 32, 'latent_samples': 64,
                 'channel_trials': 3, 'val_trials': 3, 'calibration_ps': [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]}
    }
    PROFILES['benchmark'] = {**PROFILES['full'], 'steps1': 200, 'steps2': 200, 'replicates': 1, 'val_every': 50}
    PROFILES['reconstruct'] = {**PROFILES['pilot'], 'grid': 24, 'sprite': 14, 'shift': 0,
                               'batch_size': 2, 'pool_size': 16, 'steps1': 6000, 'steps2': 0,
                               'rollout_min': 40, 'rollout_max': 48, 'eval_steps': 48,
                               'val_every': 100, 'replicates': 1, 'eval_cases': 1, 'channel_trials': 1, 'val_trials': 1}
    PROFILES['reconstruct-smoke'] = {**PROFILES['smoke'], 'shift': 0, 'steps2': 0}
    MODELS = [
        {'name': '1A_det', 'family': 'message', 'message_stochastic': False, 'query_stochastic': False, 'response_stochastic': False},
        {'name': '1A_bern', 'family': 'message', 'message_stochastic': True, 'query_stochastic': False, 'response_stochastic': False},
        {'name': '1B_det', 'family': 'query_answer', 'message_stochastic': False, 'query_stochastic': False, 'response_stochastic': False},
        {'name': '1B_qbern', 'family': 'query_answer', 'message_stochastic': False, 'query_stochastic': True, 'response_stochastic': False},
        {'name': '1B_rbern', 'family': 'query_answer', 'message_stochastic': False, 'query_stochastic': False, 'response_stochastic': True},
        {'name': '1B_qrbern', 'family': 'query_answer', 'message_stochastic': False, 'query_stochastic': True, 'response_stochastic': True}
    ]
    auto_device = 'cuda' if torch.cuda.is_available() else 'cpu'
    CFG = {**PROFILES[profile], 'profile': profile, 'seed': SEED, 'models': MODELS,
           'device': auto_device if requested_device == 'auto' else requested_device,
           'resume': resume, 'cpu_threads': int(os.environ.get('NCA_CPU_THREADS', '1')), 'cue_dim': 9,
           'budget_regimes': ['updates', 'bits'], 'learning_rate': 0.001,
           'step_size': 0.15, 'state_bound': 8.0, 'grad_clip': 1.0, 'state_penalty': 0.0001,
           'foreground_weight': 4.0, 'bernoulli_floor': 0.05, 'temperature': 1.0,
           'reset_fraction': 0.25, 'damage_probability': 0.25, 'switch_probability': 0.5,
           'damage_radius_fraction': 0.15, 'train_ps': [0.0, 0.25, 0.5, 0.75, 1.0],
           'stress_flip_rates': [0.0, 0.2] if profile == 'smoke' else [0.0, 0.05, 0.1, 0.2, 0.35],
           'quality_ratio_max': 0.35, 'class_margin_min': 0.15, 'alpha_iou_min': 0.4,
           'stable_success_points': 2, 'phase2_initialization': 'phase1', 'protocol_version': 1}
    if profile in ['reconstruct', 'reconstruct-smoke']:
        control = {'name': 'continuous_control', 'family': 'continuous', 'message_stochastic': False,
                   'query_stochastic': False, 'response_stochastic': False}
        CFG.update(task='canonical_reconstruction', models=[control] + [s for s in MODELS if s['name'] in ['1A_det', '1B_det']],
                   budget_regimes=['updates'], damage_probability=0.0, switch_probability=0.0,
                   quiescent_cells=True, activation='relu', normalize_gradients=True, reset_worst=True, detail_weight=0.5,
                   fresh_seed_steps=2000, validation_horizons=[1, 2],
                   gate_foreground_rmse=0.10, gate_alpha_iou=0.85, gate_detail_ratio=0.75,
                   stress_flip_rates=[0.0], protocol_version=2)
    return CFG


In [ ]:
# study matrix; widths are hidden units, not transmitted bits; private state stays fixed
STUDY_PROFILES = ['study-horseshoe-controls', 'study-horseshoe-controls-smoke', 'study-crab-controls', 'study-crab-controls-smoke', 'study-packet-conditioning', 'study-packet-conditioning-smoke', 'study-reliability', 'study-reliability-smoke', 'study-sprite-smoke', 'study-quantization', 'study-quantization-smoke', 'study-smoke', 'study-pilot', 'study-clean', 'study-full', 'study-matched', 'study-controls', 'study-controls-smoke', 'study-conditioning', 'study-conditioning-smoke']

def study_parameter_count(family, width, dim, state_dim=16, cue_dim=9):
    own = state_dim + cue_dim
    linear = lambda a, b: (a + 1) * b
    update = linear(own + (8 if family == 'message' else 24) * dim, width) + linear(width, state_dim)
    if family == 'message':
        return linear(own + 2, width) + linear(width, dim) + update
    return (linear(own + 2, width) + linear(width, dim) + linear(own + 2 + dim, width)
            + linear(width, width) + linear(width, width) + linear(width, dim) + update)

def study_models(widths, message_sizes, mode, include_reliability, train_rates, state_dim=16, cue_dim=9):
    models = []
    configurations = [('message', 'det', False, False, False), ('query_answer', 'det', False, False, False)]
    if include_reliability:
        configurations += [('message', 'mbern', True, False, False), ('query_answer', 'qbern', False, True, False),
                           ('query_answer', 'rbern', False, False, True), ('query_answer', 'qrbern', False, True, True)]
    def add(family, width, bits, reliability='det', flags=(False, False, False), kind='binary', flips=None, extra=False):
        target = study_parameter_count('message', width, bits, state_dim, cue_dim)
        actual = min(range(4, 513), key=lambda w: abs(study_parameter_count(family, w, bits, state_dim, cue_dim) - target)) if mode == 'parameters' and kind in ['binary', 'float'] else width
        name = f'{family}__w{width:03d}__b{bits}__{kind}__{reliability}__{mode}'
        models.append({'name': name, 'family': family, 'study_spec': True, 'requested_width': width, 'width': actual,
                       'message_dim': bits, 'message_bits': bits, 'kind': kind, 'capacity_mode': mode,
                       'parameter_target': target, 'parameters_expected': study_parameter_count(family, actual, bits, state_dim, cue_dim) if kind != 'continuous' else (state_dim + cue_dim + 8 * state_dim + 1) * actual + (actual + 1) * state_dim, 'reliability': reliability, 'message_stochastic': flags[0],
                       'query_stochastic': flags[1], 'response_stochastic': flags[2], 'bernoulli_floor': 0.05,
                       'train_flips': {} if flips is None else flips, 'extra_payload_control': extra,
                       'allowed_regimes': ['updates', 'bits'] if kind == 'binary' else ['updates']})
    for width in widths:
        for bits in message_sizes:
            for family, label, m, q, r in configurations:
                add(family, width, bits, label, (m, q, r))
            if include_reliability:
                for rate in train_rates:
                    for family, scope in [('message', 'm'), ('query_answer', 'q'), ('query_answer', 'r'), ('query_answer', 'qr')]:
                        add(family, width, bits, f'{scope}flip{round(1000 * rate):03d}', flips={ch: rate for ch in scope})
            for family in ['message', 'query_answer']:
                add(family, width, bits, kind='float')
        # message(2b) and query/answer(b+b) transmit equal bits at equal state-update counts
        for bits in sorted({2 * b for b in message_sizes} - set(message_sizes)):
            add('message', width, bits, extra=True)
        add('message', width, 1, kind='none')
        add('continuous', width, 1, kind='continuous')
    return models

def study_settings(profile, requested_device, resume=False):
    if profile in ['study-crab-controls', 'study-crab-controls-smoke', 'study-horseshoe-controls', 'study-horseshoe-controls-smoke']:
        return crab_control_settings(profile, requested_device, resume)
    if profile == 'study-sprite-smoke':
        return sprite_settings(profile, requested_device, resume)
    packet_conditioning = profile in ['study-packet-conditioning', 'study-packet-conditioning-smoke']
    reliability = profile in ['study-reliability', 'study-reliability-smoke']
    quantization = profile in ['study-quantization', 'study-quantization-smoke']
    conditioning = profile in ['study-conditioning', 'study-conditioning-smoke']
    controls = conditioning or profile in ['study-controls', 'study-controls-smoke']
    smoke = profile == 'study-packet-conditioning-smoke' or profile == 'study-reliability-smoke' or profile == 'study-quantization-smoke' or profile in ['study-smoke', 'study-controls-smoke', 'study-conditioning-smoke']
    pilot = profile == 'study-pilot'
    widths = [128] if quantization or reliability or packet_conditioning else [64, 128] if controls else [32] if smoke else [32, 64, 128]
    sizes, state_dim = [1, 2, 4], 16
    mode = 'parameters' if profile == 'study-matched' else 'width'
    models = study_models(widths, sizes, mode, profile != 'study-clean', [0.1] if smoke or pilot else [0.05, 0.2], state_dim=state_dim)
    if controls:
        models = [m for m in models if m['kind'] == 'continuous' or m['kind'] == 'float' and m['message_dim'] == 4]
    cfg = experiment_settings('smoke' if smoke else 'pilot', requested_device, resume)
    cfg.update(task='communication_study', protocol_version=4, state_dim=state_dim, models=models,
               widths=widths, message_sizes=sizes, capacity_mode=mode, profile=profile, steps2=0,
               grid=24 if smoke else 32, sprite=10 if smoke else 14, shift=2 if smoke else 3,
               steps1=2 if smoke else 200 if pilot else 6000, batch_size=2 if smoke else 4,
               pool_size=8 if smoke else 32, replicates=1 if smoke or pilot else 5,
               rollout_min=8 if smoke else 48, rollout_max=8 if smoke else 64,
               rollout_quantum=8, eval_steps=8 if smoke else 64, val_every=1 if smoke else 50 if pilot else 200,
               eval_cases=1 if smoke else 4 if pilot else 16, channel_trials=1 if smoke or pilot else 3,
               val_trials=1 if smoke or pilot else 2, validation_horizons=[1, 2],
               train_rates=[0.1] if smoke or pilot else [0.05, 0.2], stress_flip_rates=[0.0, 0.2] if smoke else [0.0, 0.05, 0.1, 0.2],
               quiescent_cells=True, activation='tanh', normalize_gradients=True, reset_worst=True,
               fresh_seed_steps=1 if smoke else 100 if pilot else 2000, detail_weight=0.5,
               damage_probability=0.0, switch_probability=0.0, budget_regimes=['updates', 'bits'],
               gate_foreground_rmse=0.10, gate_alpha_iou=0.85, gate_detail_ratio=0.75, val_success_min=0.8)
    if controls:
        cfg.update(control_diagnostic=True, replicates=1, budget_regimes=['updates'], channel_trials=1, val_trials=1,
                   train_rates=[], stress_flip_rates=[0.0], message_sizes=[4])
    if conditioning:
        base = next(m for m in models if m['kind'] == 'continuous' and m['width'] == 128)
        models = [{**base, 'name': base['name'] + '__cue_' + mode, 'cue_mode': mode} for mode in ['seed', 'broadcast']]
        cfg.update(conditioning_diagnostic=True, models=models, widths=[128])
    if quantization:
        models = study_models([128], sizes, 'width', False, [], state_dim=state_dim)
        models = [{**m, 'cue_mode': 'seed', 'allowed_regimes': ['updates']} for m in models
                  if m['kind'] in ['binary', 'float', 'continuous'] and not m['extra_payload_control']]
        cfg.update(quantization_study=True, models=models, replicates=3, budget_regimes=['updates'],
                   channel_trials=1, val_trials=1, train_rates=[], stress_flip_rates=[0.0])
    if reliability:
        models = study_models([128], [4], 'width', True, [0.1], state_dim=state_dim)
        models = [{**m, 'cue_mode': 'seed', 'allowed_regimes': ['updates']} for m in models
                  if m['kind'] in ['binary', 'float', 'continuous'] and not m['extra_payload_control']]
        cfg.update(reliability_study=True, models=models, message_sizes=[4], replicates=1 if smoke else 3,
                   budget_regimes=['updates'], train_rates=[0.1], stress_flip_rates=[0.0, 0.1] if smoke else [0.0, 0.05, 0.1, 0.2],
                   channel_trials=2 if smoke else 3, val_trials=2 if smoke else 3)
    if packet_conditioning:
        base = study_models([128], [4], 'width', False, [], state_dim=state_dim)
        models = [{**m, 'name': m['name'] + '__cue_' + cue_mode, 'cue_mode': cue_mode, 'allowed_regimes': ['updates']}
                  for m in base if m['kind'] in ['binary', 'float', 'continuous'] and not m['extra_payload_control']
                  for cue_mode in ['seed', 'broadcast']]
        cfg.update(packet_conditioning_study=True, models=models, message_sizes=[4], replicates=1 if smoke else 3,
                   budget_regimes=['updates'], channel_trials=1, val_trials=1, train_rates=[], stress_flip_rates=[0.0])
    return cfg

# execution-only sprite profile; future learning profiles can reuse the target contract
def sprite_settings(profile, requested_device, resume=False):
    sprite_id = os.environ.get('NCA_SPRITE_ID', '')
    matches = [s for s in SPRITE_MANIFEST['sprites'] if s['id'] == sprite_id]
    if len(matches) != 1:
        raise ValueError('select --sprite horseshoe_crab or --sprite green_crab')
    cfg = study_settings('study-controls-smoke', requested_device, resume)
    cfg.update(profile=profile, target=dict(matches[0]), grid=112, sprite=64, shift=4, widths=[128],
               models=[s for s in cfg['models'] if s['kind'] == 'continuous' and s['width'] == 128])
    return cfg

# bounded green-crab learning check; reuse paired conditioning reports, no binary/reliability sweep
def crab_control_settings(profile, requested_device, resume=False):
    horseshoe = profile in ['study-horseshoe-controls', 'study-horseshoe-controls-smoke']
    sprite_id = 'horseshoe_crab' if horseshoe else 'green_crab'
    if os.environ.get('NCA_SPRITE_ID', '') != sprite_id:
        raise ValueError('this profile requires --sprite ' + sprite_id)
    smoke = profile.endswith('-smoke')
    cfg = study_settings('study-packet-conditioning-smoke' if smoke else 'study-packet-conditioning', requested_device, resume)
    target = next(s for s in SPRITE_MANIFEST['sprites'] if s['id'] == sprite_id)
    cfg.update(profile=profile, target=dict(target), crab_control_diagnostic=True, grid=112, sprite=64, shift=4,
               models=[m for m in cfg['models'] if m['kind'] in ['float', 'continuous']], replicates=1,
               batch_size=2, pool_size=16, eval_cases=1 if smoke else 8, checkpoint_rollout=True,
               rollout_min=8 if smoke else 64, rollout_max=8 if smoke else 96, rollout_quantum=8 if smoke else 16,
               eval_steps=8 if smoke else 96)
    if horseshoe:
        cfg['models'] = [m for m in cfg['models'] if m['kind'] == 'continuous']
    return cfg

def study_jobs(cfg):
    return [{'model': spec['name'], 'regime': regime, 'replicate': replicate,
             'run': f"{regime}__{spec['name']}__r{replicate}"}
            for regime in cfg['budget_regimes'] for replicate in range(cfg['replicates']) for spec in cfg['models']
            if regime in spec.get('allowed_regimes', cfg['budget_regimes'])]


In [ ]:
# globals; edit this cell for a new experiment, then rerun the whole pipeline
ROOT = Path(os.environ['NCA_PROJECT_ROOT'])
TASK = ROOT / 'a_import'
INPUT, OUTPUT = TASK / 'input', TASK / 'output'
PROFILE = os.environ['NCA_PROFILE']
if PROFILE in ['study-sprite-smoke', 'study-crab-controls', 'study-crab-controls-smoke', 'study-horseshoe-controls', 'study-horseshoe-controls-smoke']:
    SPRITE_MANIFEST = json.loads((INPUT / 'sprites/manifest.json').read_text())
settings = study_settings if PROFILE in STUDY_PROFILES else experiment_settings
CFG = settings(PROFILE, os.environ['NCA_DEVICE'], os.environ['NCA_RESUME'] == '1')
SEED = CFG['seed']
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(CFG['cpu_threads'])
torch.use_deterministic_algorithms(True)

In [ ]:
# source images; authored geometric fixtures, no font/network/download dependency
def fixture(kind):
    image = Image.new('RGBA', (128, 128), (0, 0, 0, 0))
    draw = ImageDraw.Draw(image)
    dark = (35, 31, 39, 255)
    if kind == 'cat':
        draw.polygon([(17, 58), (15, 9), (48, 34), (83, 34), (115, 9), (111, 61)], fill=(225, 143, 48, 255), outline=dark, width=4)
        draw.polygon([(23, 38), (22, 21), (39, 36)], fill=(246, 148, 156, 255))
        draw.polygon([(91, 36), (107, 21), (105, 39)], fill=(246, 148, 156, 255))
        draw.ellipse((14, 29, 115, 119), fill=(241, 174, 66, 255), outline=dark, width=4)
        draw.ellipse((27, 58, 50, 79), fill=(101, 158, 128, 255), outline=dark, width=3)
        draw.ellipse((80, 58, 103, 79), fill=(101, 158, 128, 255), outline=dark, width=3)
        draw.line((39, 62, 39, 74), fill=dark, width=4)
        draw.line((91, 62, 91, 74), fill=dark, width=4)
        draw.polygon([(57, 83), (73, 83), (65, 92)], fill=(178, 88, 107, 255))
        draw.arc((45, 87, 66, 104), 0, 160, fill=dark, width=3)
        draw.arc((65, 87, 86, 104), 20, 180, fill=dark, width=3)
        for y in [85, 94]:
            draw.line((7, y - 4, 43, y), fill=dark, width=3)
            draw.line((87, y, 123, y - 4), fill=dark, width=3)
    else:
        draw.ellipse((10, 10, 118, 118), fill=(250, 211, 65, 255), outline=dark, width=4)
        draw.ellipse((35, 41, 46, 59), fill=dark)
        draw.ellipse((82, 41, 93, 59), fill=dark)
        draw.arc((33, 50, 96, 100), 5, 175, fill=dark, width=7)
        draw.ellipse((23, 67, 38, 76), fill=(241, 144, 106, 255))
        draw.ellipse((90, 67, 105, 76), fill=(241, 144, 106, 255))
    return image

In [ ]:
# import boundary; validate only here, then trust downstream inputs
assets = OUTPUT / 'assets'
assets.mkdir(exist_ok=True)
image_rows = []
if 'target' in CFG:
    target = CFG['target']
    supplied = INPUT / 'sprites' / target['file']
    image = Image.open(supplied).convert('RGBA')
    array = np.asarray(image)
    if image.size != (target['width'], target['height']) or image.size != (64, 64):
        raise ValueError('sprite must have a 64x64 canvas; no implicit resizing')
    if not np.any(array[..., 3] > 0) or not np.any(array[..., 3] < 255):
        raise ValueError('sprite requires foreground and transparent padding')
    canvas = Image.new('RGBA', (CFG['grid'], CFG['grid']), (0, 0, 0, 0))
    corner = ((CFG['grid'] - image.width) // 2, (CFG['grid'] - image.height) // 2)
    canvas.paste(image, corner)  # preserve source RGBA bytes, no resampling
    # existing tensor format has two class slots; Phase 1 uses slot zero only
    for name in ['cat', 'smiley']:
        canvas.save(assets / f'{name}.png')
    target.update(source_sha256=hashlib.sha256(supplied.read_bytes()).hexdigest(),
                  canvas_sha256=hashlib.sha256((assets / 'cat.png').read_bytes()).hexdigest())
    image_rows.append({'name': target['id'], 'origin': 'provided sprite', **target})
    CFG['models'] = [{**spec, **{'target_' + key: target[key] for key in ['id', 'role', 'source_sha256', 'canvas_sha256']}} for spec in CFG['models']]
    pd.DataFrame([target]).to_csv(OUTPUT / 'sprite_manifest.csv', index=False)
else:
    for name in ['cat', 'smiley']:
        supplied = INPUT / f'{name}.png'
        image = Image.open(supplied).convert('RGBA') if supplied.is_file() else fixture(name)
        array = np.asarray(image)
        if not np.any(array[..., 3] > 0):
            raise ValueError(f'{name}.png has no foreground')
        image = ImageOps.contain(image, (CFG['sprite'], CFG['sprite']), Image.Resampling.LANCZOS)
        canvas = Image.new('RGBA', (CFG['grid'], CFG['grid']), (0, 0, 0, 0))
        corner = (CFG['grid'] // 2 - image.width // 2, CFG['grid'] // 2 - image.height // 2)
        canvas.alpha_composite(image, corner)
        canvas.save(assets / f'{name}.png')
        image_rows.append({'name': name, 'origin': 'provided' if supplied.is_file() else 'authored geometric fixture',
                           'sha256': hashlib.sha256((assets / f'{name}.png').read_bytes()).hexdigest()})
if CFG['grid'] <= 2 * CFG['shift'] + np.sqrt(2) * CFG['sprite']:
    raise ValueError('grid too small for the requested translated/rotated sprite')
if not 0 <= CFG['bernoulli_floor'] < 0.5:
    raise ValueError('Bernoulli floor must be in [0, 0.5)')
display(pd.DataFrame(image_rows))

In [ ]:
if CFG.get('task') != 'communication_study':
    # experimental contract; seed-only continuous cues, binary-only cell-to-cell access
    design = pd.DataFrame([
        ('state', 'RGBA + private hidden channels; every grid site computes synchronously'),
        ('seed cue', 'one site only: presence, cos(theta), sin(theta), saturation, RGB gains, p, u'),
        ('translation', 'move the seed; no absolute coordinates enter the model'),
        ('rotation/color', 'conditioned reconstruction and held-out transformation generalization; not guaranteed group invariance'),
        ('messages', 'one hard bit per directed edge; direction-aware addressed inbox with eight Moore ports'),
        ('query/answer', 'requester sends q; neighbor receiver reads local state + q; neighbor sends one-bit r back'),
        ('no bypass', 'six binary designs: no neighbor RGBA/hidden reads, spatial convolutions, global cue broadcast or neighbor alpha gating'),
        ('hard training', 'hard threshold or Bernoulli in forward pass; biased sigmoid straight-through gradients'),
        ('Bernoulli', 'p_bit = floor + (1 - 2*floor)*sigmoid(logit / temperature); floor keeps uncertainty from vanishing'),
        ('external errors', 'independent bit flips are evaluation interventions, separate from Bernoulli generation'),
        ('budget regimes', 'same update counts or same directed-bit counts; parameter counts approximately matched'),
        ('Phase 2 choice', 'target = smiley iff u < p; only p and u enter the seed, never the chosen class'),
        ('response', 'change seed p/u in a running state; also test damage repair and persistence'),
        ('limits', 'two known templates; no claim of novel categories, autonomous evolution or VAE/GAN training'),
        ('replicates', 'all generators initialized at 63; disjoint jumped/advanced streams, paired across configurations'),
        ('validation', 'checkpoint selection uses validation only; hold-out tests never affect optimizer or selection')
    ], columns=['item', 'definition'])
    design.to_csv(OUTPUT / 'experimental_contract.csv', index=False)
    display(design)
    if CFG.get('task') == 'canonical_reconstruction':
        canonical_contract = pd.DataFrame([
            ('scope', 'one centered upright full-color cat; Phase 1 only; no claim of held-out-image or transformation generalization'),
            ('control', 'continuous_control reads eight float32 neighbor state vectors; explicit exception to the binary-only contract; counted at 32 bits per scalar'),
            ('quiet background', 'zero-centered update MLP keeps zero-input cells quiet and retains gradients at zero-valued inboxes; no neighbor activity mask is transmitted'),
            ('detail objective', 'foreground interior RGB gradient error supplements weighted pixel loss'),
            ('training', 'first 2000 steps from fresh seeds, then pool; no damage; ReLU MLPs; normalize per-parameter gradients before clipping'),
            ('quality gate', 'foreground RMSE <= 0.10, alpha IoU >= 0.85, and detail RMSE <= 0.75 times flat-color oracle-mask baseline'),
            ('next step', 'inspect examples and persistence before adding transformations, stochastic channels or a second image')
        ], columns=['item', 'definition'])
        canonical_contract.to_csv(OUTPUT / 'reconstruction_contract.csv', index=False)
        display(canonical_contract)


In [ ]:
# provenance and configuration
sources = pd.DataFrame([
    ('NCA growth/persistence/repair', 'https://distill.pub/2020/growing-ca/'),
    ('local conditional response', 'https://arxiv.org/abs/2305.12971'),
    ('straight-through estimator caveat', 'https://arxiv.org/abs/1308.3432'),
    ('reproducibility limits', 'https://docs.pytorch.org/docs/stable/notes/randomness.html'),
    ('Bernoulli sampling API', 'https://docs.pytorch.org/docs/stable/generated/torch.bernoulli.html')
], columns=['topic', 'url'])
sources.to_csv(OUTPUT / 'sources.csv', index=False)
packages = ['torch', 'numpy', 'pandas', 'seaborn', 'matplotlib', 'pillow', 'nbformat', 'IPython', 'scipy']
environment = {name: importlib.metadata.version(name) for name in packages}
environment.update({'python': sys.version, 'platform': platform.platform(), 'device': CFG['device']})
(OUTPUT / 'environment.json').write_text(json.dumps(environment, indent=2))
(OUTPUT / 'image_provenance.json').write_text(json.dumps(image_rows, indent=2))
(OUTPUT / 'config.json').write_text(json.dumps(CFG, indent=2))
display(pd.DataFrame([{'profile': PROFILE, 'device': CFG['device'], 'seed': SEED,
                       'training_runs': len(study_jobs(CFG)) * (1 if CFG['steps2'] == 0 else 2)}]))

In [ ]:
# planned interventions and run identities; no Phase 2 in this study
if CFG.get('task') == 'communication_study':
    design = pd.DataFrame(CFG['models'])
    design.drop(columns=['train_flips', 'allowed_regimes']).to_csv(OUTPUT / 'study_design.csv', index=False)
    contract = pd.DataFrame([
        ('task', 'one ' + CFG.get('target', {}).get('id', 'cat') + ', transformed rotation/translation/color conditions; seed-local controls; no class switching'),
        ('scope', 'conditioned reproduction and held-out transformation generalization, not guaranteed group equivariance'),
        ('width', '32/64/128 hidden units; study-matched instead matches parameters to the corresponding outbound model'),
        ('state', '16 private channels in every study profile, including smoke; no neighbor alive-mask side channel'),
        ('payload', '1/2/4 independent binary components per directed port; an 8-bit outbound control matches 4+4-bit query/reply'),
        ('reliability', 'learned Bernoulli floor=0.05 and independent per-bit channel flips are distinct interventions; floor already enforces uncertainty'),
        ('controls', 'same-shape float32 messages/queries/replies; no-message and full continuous-neighbor controls; floats tested only at equal update counts'),
        ('budgets', 'equal actual state updates or equal total directed transmitted bits; query and reply both charged; padding edges excluded'),
        ('replicates', 'all seed 63; independent replicate streams; cells, cases and channel trials are not independent training replicates'),
        ('quality', 'foreground RMSE, alpha IoU, interior detail vs flat oracle-mask baseline; report pass and failure fractions'),
        ('selection', 'validation losses under training-channel conditions only; test corruption never selects checkpoints'),
        ('interpretation', 'same-width protocols have different parameter counts; matched-capacity profile is a separate sensitivity analysis')
    ], columns=['item', 'definition'])
    if CFG.get('conditioning_diagnostic', False):
        contract.loc[contract['item'].eq('task'), 'definition'] = 'conditioning diagnostic: continuous-neighbor, width 128, seed-only versus broadcast transform cues; no binary comparison'
        contract.loc[contract['item'].eq('payload'), 'definition'] = 'continuous neighbor state; broadcast variant additionally receives six global transform channels each update, excluded from neighbor bit accounting'
        contract.loc[contract['item'].eq('interpretation'), 'definition'] = 'broadcast changes information access and can activate distant cells; exclude it from bandwidth/reliability conclusions and seed-cone bounds'
    if CFG.get('quantization_study', False):
        contract.loc[contract['item'].eq('scope'), 'definition'] = 'clean-channel quantization study: same-shape binary versus float, equal updates, seed-only cues, 3 training replicates; held-out generalization reported separately'
    if CFG.get('packet_conditioning_study', False):
        contract.loc[contract['item'].eq('task'), 'definition'] = 'cat conditioning control: seed-only versus broadcast transform cues for binary/float outbound and query/reply, plus continuous-neighbor baseline'
        contract.loc[contract['item'].eq('width'), 'definition'] = '128 hidden units; all models retain 16 private channels and identical input dimensions between cue modes'
        contract.loc[contract['item'].eq('payload'), 'definition'] = 'four binary bits or four float32 components per directed packet; continuous-neighbor state is a separate baseline'
        contract.loc[contract['item'].eq('controls'), 'definition'] = 'five architectures, each with fresh seed and broadcast counterparts; matched initial weights within replicate; clean deterministic channels only'
        contract.loc[contract['item'].eq('budgets'), 'definition'] = 'equal state-update budget; broadcast supplies six global transform channels each update, excluded from neighbor payload accounting; not an equal-information comparison'
        contract.loc[contract['item'].eq('interpretation'), 'definition'] = 'broadcast is an information-access bypass and can activate distant cells; keep separate from bandwidth/reliability rankings and seed-cone bounds'
    if CFG.get('crab_control_diagnostic', False):
        contract.loc[contract['item'].eq('task'), 'definition'] = CFG['target']['id'] + ' at native 64x64 resolution on a 112x112 grid; clean controls with paired seed/broadcast cues'
        contract.loc[contract['item'].eq('controls'), 'definition'] = str(len(CFG['models'])) + ' architecture/cue combinations; one seed-63 training replicate, fresh identical initial weights within cue pairs; clean channels, no binary comparison'
        contract.loc[contract['item'].eq('payload'), 'definition'] = 'four float32 components per directed packet; full continuous-neighbor state is a separate baseline'
        contract.loc[contract['item'].eq('interpretation'), 'definition'] = 'execution completion is not learning success; one-replicate diagnostic with no architecture-ranking inference; broadcasting is extra information and excluded from communication rankings'
    if CFG['profile'].startswith('study-horseshoe-controls'):
        contract.loc[contract['item'].eq('payload'), 'definition'] = 'full continuous-neighbor state only; no learned message/query/reply packets'
    contract.to_csv(OUTPUT / 'study_contract.csv', index=False)
    contract.to_csv(OUTPUT / 'experimental_contract.csv', index=False)
    display(design[['name', 'width', 'message_bits', 'kind', 'reliability']])
    display(contract)
